In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error

df = pd.read_csv("clean_data.csv")
X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

print("Shape X_train:", X_train_scaled.shape)
print("Shape X_test:", X_test_scaled.shape)

Shape X_train: (5730, 5)
Shape X_test: (1433, 5)


In [ ]:
ols = LinearRegression()
ols.fit(X_train_scaled, y_train)
y_pred_ols = ols.predict(X_test_scaled)
rmse_ols = np.sqrt(mean_squared_error(y_test, y_pred_ols))

alphas_ridge = np.logspace(-3, 3, 200)
ridge = RidgeCV(alphas=alphas_ridge, cv=5)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

alphas_lasso = np.logspace(-4, 2, 200)
lasso = LassoCV(alphas=alphas_lasso, cv=5, max_iter=10000, random_state=51)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

print(f"Best Alpha cho Ridge: {ridge.alpha_:.4f}")
print(f"Best Alpha cho LASSO: {lasso.alpha_:.4f}")

Best Alpha cho Ridge: 10.9699
Best Alpha cho LASSO: 0.0784


In [19]:
# Tạo dữ liệu cho bảng tổng hợp
table_rows = []

# Dòng 1: Best Alpha
table_rows.append({
    "w": "Best Alpha",
    "Ordinary Least Squares": "-",
    "Ridge": f"{ridge.alpha_:.4f}",
    "LASSO": f"{lasso.alpha_:.4f}"
})

# Dòng 2: Intercept w0
table_rows.append({
    "w": "0",
    "Ordinary Least Squares": f"{ols.intercept_:.4f}",
    "Ridge": f"{ridge.intercept_:.4f}",
    "LASSO": f"{lasso.intercept_:.4f}"
})

# Các dòng đặc trưng (weights w_i)
for i, col_name in enumerate(X.columns):
    l_coef = lasso.coef_[i]
    # Nếu trọng số của LASSO bị ép về 0 thì để trống giống bảng mẫu
    l_str = f"{l_coef:.4f}" if abs(l_coef) > 1e-4 else ""
    
    table_rows.append({
        "w": col_name,
        "Ordinary Least Squares": f"{ols.coef_[i]:.4f}",
        "Ridge": f"{ridge.coef_[i]:.4f}",
        "LASSO": l_str
    })

# Dòng cuối: Test RMSE
table_rows.append({
    "w": "Test RMSE",
    "Ordinary Least Squares": f"{rmse_ols:.4f}",
    "Ridge": f"{rmse_ridge:.4f}",
    "LASSO": f"{rmse_lasso:.4f}"
})

result_table = pd.DataFrame(table_rows)

# Định dạng màu sắc pastel dịu mắt, hài hòa, dễ nhìn
def highlight_footer(row):
    if row["w"] == "Test RMSE":
        # Màu cam đất/be pastel ấm dịu, chữ nâu đậm tương phản tốt
        return ['background-color: #fceddf; color: #6e3314; font-weight: bold'] * len(row)
    elif row["w"] == "Best Alpha":
        # Màu xám xanh pastel nhẹ nhàng, mát mắt
        return ['background-color: #edf2f7; color: #1a202c; font-weight: bold'] * len(row)
    return [''] * len(row)

styled_table = result_table.style.apply(highlight_footer, axis=1)\
                           .set_properties(**{"text-align": "right", "padding": "6px 14px"}, subset=["Ordinary Least Squares", "Ridge", "LASSO"])\
                           .set_properties(**{"text-align": "left", "padding": "6px 14px"}, subset=["w"])\
                           .set_table_styles([
                               {'selector': 'th', 'props': [('background-color', '#4a5568'), ('color', '#ffffff'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px 14px')]},
                               {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('border', '1px solid #cbd5e1')]},
                               {'selector': 'td', 'props': [('border-bottom', '1px solid #f1f5f9')]}
                           ])\
                           .hide(axis='index')

styled_table


w,Ordinary Least Squares,Ridge,LASSO
Best Alpha,-,10.9699,0.0784
0,74.0524,74.0524,74.0524
bath,0.0484,0.0734,
balcony,-0.8148,-0.8044,-0.7122
total_sqft_float,20.9867,20.9211,20.8126
bhk,-2.1352,-2.0838,-1.8877
price_per_sqft,33.3791,33.2836,33.2162
Test RMSE,9.1976,9.1973,9.1910
